In [1]:
!pip install neo4j

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.0/203.0 kB 2.7 MB/s eta 0:00:00a 0:00:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 505.5/505.5 kB 9.5 MB/s eta 0:00:00ta 0:00:01
  Created wheel for neo4j: filename=neo4j-5.20.0-py3-none-any.whl size=280771 sha256=8b885631039b5558f56995c94b52239921c9b0eb6291994540f177b0ac59ed2c
  Stored in directory: /Users/anhthubui/Library/Caches/pip/wheels/aa/7d/7c/d47bc6347b27804958165fa526d1b50a0c76bae5545c866f99
Successfully built neo4j


In [3]:
pip install pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.8/114.8 kB 2.7 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 29.9 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.0/14.0 MB 36.0 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 345.4/345.4 kB 22.2 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [1]:
import pandas as pd
from neo4j import GraphDatabase
import numpy as np

# Course data
- Module and Presentation Identification: Each module is identified by a unique code (code_module), 
- and each presentation by a year-based code (code_presentation) indicating February (B) or October (J) starts
- Presentation Length: The module_presentation_length column specifies the duration of each module-presentation in days

In [3]:
courses = pd.read_csv('./archive/courses.csv')
display(courses.head())

,code_module,code_presentation,module_presentation_length
0,AAA,2013J,268
1,AAA,2014J,269
2,BBB,2013J,268
3,BBB,2014J,262
4,BBB,2013B,240


In [4]:
courses.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 22 entries, 0 to 21
Data columns (total 3 columns):
 #   Column                      Non-Null Count  Dtype 
---  ------                      --------------  ----- 
 0   code_module                 22 non-null     object
 1   code_presentation           22 non-null     object
 2   module_presentation_length  22 non-null     int64 
dtypes: int64(1), object(2)
memory usage: 660.0+ bytes


In [5]:
# all courses
list = ['code_module', 'code_presentation', 'module_presentation_length']
for i in list:
    print(courses[i].unique())

['AAA' 'BBB' 'CCC' 'DDD' 'EEE' 'FFF' 'GGG']
['2013J' '2014J' '2013B' '2014B']
[268 269 262 240 234 241 261]


# Assessment Data 

- Assessment Details: Each assessment is linked to a module (code_module) and presentation (code_presentation), and identified by a unique assessment number (id_assessment).
- Assessment Types and Dates: Assessments are categorized into TMA, CMA, and Exams (assessment_type). The date column indicates the final submission date, calculated as days since the module-presentation start.
- Weighting System: The weight column represents the assessment's weight in percentage. Exams are separately weighted at 100%, while the sum of other assessments' weights totals 100%.

In [6]:
assessments = pd.read_csv('./archive/assessments.csv')
display(assessments.head())

,code_module,code_presentation,id_assessment,assessment_type,date,weight
0,AAA,2013J,1752,TMA,19.0,10.0
1,AAA,2013J,1753,TMA,54.0,20.0
2,AAA,2013J,1754,TMA,117.0,20.0
3,AAA,2013J,1755,TMA,166.0,20.0
4,AAA,2013J,1756,TMA,215.0,30.0


In [8]:
# checking the unique values of the dataframe
list = ['code_module', 'code_presentation', 'assessment_type', 'date', 'weight']
for i in list:
    print(assessments[i].unique())

['AAA' 'BBB' 'CCC' 'DDD' 'EEE' 'FFF' 'GGG']
['2013J' '2014J' '2013B' '2014B']
['TMA' 'Exam' 'CMA']
[ 19.  54. 117. 166. 215.  nan  89. 124. 159. 187.  47.  96. 131. 208.
  82. 152. 194.  12.  40. 110. 201.  18.  67. 137. 207.  32. 102. 151.
 200. 144. 214. 109. 158.  23.  51.  79. 114. 149. 170. 206.  25.  53.
  81. 116. 240.  88. 123. 165. 261.  74. 241.  20.  41.  62. 111. 146.
 195.  33.  68. 235. 228. 222. 236. 173. 227.  24.  52.  87. 129. 171.
  94. 136. 199. 229.  61.]
[ 10.   20.   30.  100.    1.    5.   18.    0.   35.    2.    7.    8.
   9.   22.    3.    4.    6.    7.5  12.5  15.   17.5  25.   16.   28. ]


# Student Asessment Data

- Assessment Results: Records student assessments, where each assessment is identified by its unique assessment number (id_assessment).

- Student Identification: Each student is uniquely identified by id_student, allowing tracking of individual performance.

- Submission Details: date_submitted indicates the date of student submission, measured as days since the start of the module presentation.

- Assessment Status: is_banked is a status flag indicating whether the assessment result has been transferred from a previous presentation.

- Scoring System: score represents the student's score in the assessment, ranging from 0 to 100. Scores below 40 are interpreted as Fail.

In [8]:
student_assessment = pd.read_csv('./archive/studentAssessment.csv')
display(student_assessment.head())

,id_assessment,id_student,date_submitted,is_banked,score
0,1752,11391,18,0,78.0
1,1752,28400,22,0,70.0
2,1752,31604,17,0,72.0
3,1752,32885,26,0,69.0
4,1752,38053,19,0,79.0


In [9]:
# checking the unique values of the dataframe
list = ['date_submitted', 'is_banked', 'score'] 
for i in list:    
    print(student_assessment[i].unique())

[ 18  22  17  26  19  20   9  21  16  30  32  10  25  15  54  24  33  27
  23  37  29   7  58  12  14  50  36  56  53  51  52  64  61  70 106  57
  59  48  62  55  68  69  67  63  49  47  75  60  95  65  90  66 116  42
  72  92 114 146 117 115 112 120 124 121 111 110 122  85 139 123 130 113
 118 135 127  78 134 108 126 107 119 131 138 125 100 102  94 133 128 164
 165 181 166 173 161 170 157 171 163 177 183 172 168 180 158 159 179 153
 175 169 176 174 152 156 150 162 167 178 187 188 160 215 213 212 219 214
 216 217 218 209 211 220 203 221 223 208 238 207 222 227 198 202 239 210
 204 201 194  -1  13  39   5  28  31  38  40  11  45  84  74  35  71  46
  44  93 109 132 144 129 137  79 136 185 184 155 237 224 234 205 235 226
  -4   6  41  -5  77  -3  43  91  34   8   1   3   4  -6  97  80  88  86
  81  76 186  83  89 104  87  99 101  96  98 103  82 105 193 141 142 145
 140 147 148 143 154 151 149 189 191 197 182 192 195 199 190 206 200 196
   0   2  -2  -9 228  73 229 236 -11  -7 270 233 23

# Student Info Data

- Module and Presentation Details: Each student is registered on a module identified by code_module and code_presentation, indicating the module and presentation they are enrolled in.

- Student Information: Demographic details include id_student, gender, region, highest_education, imd_band (Index of Multiple Deprivation band), age_band, num_of_prev_attempts (number of previous attempts), studied_credits, and disability status.

- Final Outcome: final_result denotes the student's final result in the module-presentation, indicating their performance outcome.

In [14]:
student_info = pd.read_csv('./archive/studentInfo2.csv')
display(student_info.head())

,code_module,code_presentation,id_student,gender,region,highest_education,imd_band,age_band,num_of_prev_attempts,studied_credits,disability,final_result
0,AAA,2013J,11391,M,East Anglian Region,HE Qualification,90-100%,55<=,0,240,N,Pass
1,AAA,2013J,28400,F,Scotland,HE Qualification,20-30%,35-55,0,60,N,Pass
2,AAA,2013J,30268,F,North Western Region,A Level or Equivalent,30-40%,35-55,0,60,Y,Withdrawn
3,AAA,2013J,31604,F,South East Region,A Level or Equivalent,50-60%,35-55,0,60,N,Pass
4,AAA,2013J,32885,F,West Midlands Region,Lower Than A Level,50-60%,0-35,0,60,N,Pass


In [15]:
student_info.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 28785 entries, 0 to 28784
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   code_module           28785 non-null  object
 1   code_presentation     28785 non-null  object
 2   id_student            28785 non-null  int64 
 3   gender                28785 non-null  object
 4   region                28785 non-null  object
 5   highest_education     28785 non-null  object
 6   imd_band              27814 non-null  object
 7   age_band              28785 non-null  object
 8   num_of_prev_attempts  28785 non-null  int64 
 9   studied_credits       28785 non-null  int64 
 10  disability            28785 non-null  object
 11  final_result          28785 non-null  object
dtypes: int64(3), object(9)
memory usage: 2.6+ MB


In [17]:
list = ['code_module', 'code_presentation', 'gender', 'region', 'highest_education', 'imd_band', 'age_band', 'num_of_prev_attempts', 'studied_credits', 'disability', 'final_result']
for i in list:
    print(student_info[i].unique())

['AAA' 'BBB' 'CCC' 'DDD' 'EEE' 'FFF' 'GGG']
['2013J' '2014J' '2013B' '2014B']
['M' 'F']
['East Anglian Region' 'Scotland' 'North Western Region'
 'South East Region' 'West Midlands Region' 'Wales' 'North Region'
 'South Region' 'Ireland' 'South West Region' 'East Midlands Region'
 'Yorkshire Region' 'London Region']
['HE Qualification' 'A Level or Equivalent' 'Lower Than A Level'
 'Post Graduate Qualification' 'No Formal quals']
['90-100%' '20-30%' '30-40%' '50-60%' '80-90%' '70-80%' nan '60-70%'
 '40-50%' '10-20' '0-10%']
['55<=' '35-55' '0-35']
[0 1 2 4 3 5 6]
[240  60 120  90 150 180 345 420 170  80  75 300 330 270 360 210 135  70
 225 585 325 130 195 105 655 165 100 390 220 160 250  30  40  45 400 235
 145 630 355  50 110 115  55  85 480 280 175  95 155 190 315 200 140 540
 310 370 205 215 255  65 430]
['N' 'Y']
['Pass' 'Withdrawn' 'Fail' 'Distinction']


there is a miswrighting in the imd_band column. Instead of 10-20% it is written 10-20

In [18]:
# remove percentage
student_info['imd_band'] = student_info['imd_band'].str.replace('10-20', '10-20%')

# Student Registration Data

- Module and Presentation Registration: Records the registration details of students for a module-presentation, identified by code_module and code_presentation.

- Student Identification: Each student is uniquely identified by id_student.

- Registration Dates: date_registration indicates the date of student registration, measured as days relative to the start of the module-presentation. Negative values denote registration before the start date.

- Unregistration Details: For students who unregistered, date_unregistration records the date of unregistration, also measured relative to the start of the module-presentation. If students completed the course, this field is empty. The final_result column in the studentInfo.csv file indicates their final outcome, with "Withdrawal" indicating unregistration.

In [20]:
student_registration = pd.read_csv('./archive/studentRegistration.csv')
display(student_registration.head())

,code_module,code_presentation,id_student,date_registration,date_unregistration
0,AAA,2013J,11391,-159.0,NaN
1,AAA,2013J,28400,-53.0,NaN
2,AAA,2013J,30268,-92.0,12.0
3,AAA,2013J,31604,-52.0,NaN
4,AAA,2013J,32885,-176.0,NaN


In [21]:
student_registration.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32593 entries, 0 to 32592
Data columns (total 5 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   code_module          32593 non-null  object 
 1   code_presentation    32593 non-null  object 
 2   id_student           32593 non-null  int64  
 3   date_registration    32548 non-null  float64
 4   date_unregistration  10072 non-null  float64
dtypes: float64(2), int64(1), object(2)
memory usage: 1.2+ MB


# Student Vle Data

- Module and Presentation Details: Interactions between students and VLE materials are recorded for each module (code_module) and presentation (code_presentation).

- Student Identification: Each student's interactions are identified by a unique id_student.

- Material Interaction Details: Records interactions with VLE materials identified by id_site.

- Interaction Timestamp: date records the date of student interaction with the material, measured as days since the start of the module-presentation.

- Interaction Frequency: sum_click denotes the number of times a student interacts with the material on a given day.

In [19]:
student_vle = pd.read_csv('./archive/studentVle.csv')
display(student_vle.head())

,code_module,code_presentation,id_student,id_site,date,sum_click
0,AAA,2013J,28400,546652,-10,4
1,AAA,2013J,28400,546652,-10,1
2,AAA,2013J,28400,546652,-10,1
3,AAA,2013J,28400,546614,-10,11
4,AAA,2013J,28400,546714,-10,1


In [20]:
student_vle.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10655280 entries, 0 to 10655279
Data columns (total 6 columns):
 #   Column             Dtype 
---  ------             ----- 
 0   code_module        object
 1   code_presentation  object
 2   id_student         int64 
 3   id_site            int64 
 4   date               int64 
 5   sum_click          int64 
dtypes: int64(4), object(2)
memory usage: 487.8+ MB


In [21]:
# check unique values of dataframe
list = ['code_module', 'code_presentation', 'id_student', 'id_site', 'date', 'sum_click']
for i in list:
    print(student_vle[i].unique())



['AAA' 'BBB' 'CCC' 'DDD' 'EEE' 'FFF' 'GGG']
['2013J' '2014J' '2013B' '2014B']
[ 28400  30268  31604 ... 676034 121182 650630]
[546652 546614 546714 ... 896948 896952 896969]
[-10  -9  -8  -7  -6  -5  -4  -3  -2  -1   0   1   2   3   4   5   6   7
   8   9  10  11  12  13  14  15  16  17  18  19  20  21  22  23  24  25
  26  27  28  29  30  31  32  33  34  35  36  37  38  39  40  41  42  43
  44  45  46  47  48  49  50  51  52  53  54  55  56  57  58  59  60  61
  62  63  64  65  66  67  68  69  70  71  72  73  74  75  76  77  78  79
  80  81  82  83  84  85  86  87  88  89  90  91  92  93  94  95  96  97
  98  99 100 101 102 103 104 105 106 107 108 109 110 111 112 113 114 115
 116 117 118 119 120 121 122 123 124 125 126 127 128 129 130 131 132 133
 134 135 136 137 138 139 140 141 142 143 144 145 146 147 148 149 150 151
 152 153 154 155 156 157 158 159 160 161 162 163 164 165 166 167 168 169
 170 171 172 173 174 175 176 177 178 179 180 181 182 183 184 185 186 187
 188 189 190 191 192 19

# Vle Data

- Material Identification: Each material is identified by a unique id_site.

- Module and Presentation Details: Materials are associated with a specific module (code_module) and presentation (code_presentation), indicating the course and its timing.

- Activity Type: activity_type specifies the role associated with the module material, such as HTML pages, PDF files, etc.

- Planned Usage Period: The week_from and week_to columns indicate the planned period during which the material is intended to be used, measured in weeks.

In [22]:
vle = pd.read_csv('./archive/vle.csv')
display(vle.head())


,id_site,code_module,code_presentation,activity_type,week_from,week_to
0,546943,AAA,2013J,resource,NaN,NaN
1,546712,AAA,2013J,oucontent,NaN,NaN
2,546998,AAA,2013J,resource,NaN,NaN
3,546888,AAA,2013J,url,NaN,NaN
4,547035,AAA,2013J,resource,NaN,NaN


In [23]:
vle.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6364 entries, 0 to 6363
Data columns (total 6 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   id_site            6364 non-null   int64  
 1   code_module        6364 non-null   object 
 2   code_presentation  6364 non-null   object 
 3   activity_type      6364 non-null   object 
 4   week_from          1121 non-null   float64
 5   week_to            1121 non-null   float64
dtypes: float64(2), int64(1), object(3)
memory usage: 298.4+ KB


# Data Cleaning
The following code contains the data cleaning steps used to prepare the dataset for the Neo4j Database

In [24]:
student_info.head()

,code_module,code_presentation,id_student,gender,region,highest_education,imd_band,age_band,num_of_prev_attempts,studied_credits,disability,final_result
0,AAA,2013J,11391,M,East Anglian Region,HE Qualification,90-100%,55<=,0,240,N,Pass
1,AAA,2013J,28400,F,Scotland,HE Qualification,20-30%,35-55,0,60,N,Pass
2,AAA,2013J,30268,F,North Western Region,A Level or Equivalent,30-40%,35-55,0,60,Y,Withdrawn
3,AAA,2013J,31604,F,South East Region,A Level or Equivalent,50-60%,35-55,0,60,N,Pass
4,AAA,2013J,32885,F,West Midlands Region,Lower Than A Level,50-60%,0-35,0,60,N,Pass


In [25]:
# remove columns highest_education, imd_band and disability, since we do not need them in our graph db
student_info = student_info.drop(columns=['highest_education', 'imd_band', 'disability'])
student_info.head()

,code_module,code_presentation,id_student,gender,region,age_band,num_of_prev_attempts,studied_credits,final_result
0,AAA,2013J,11391,M,East Anglian Region,55<=,0,240,Pass
1,AAA,2013J,28400,F,Scotland,35-55,0,60,Pass
2,AAA,2013J,30268,F,North Western Region,35-55,0,60,Withdrawn
3,AAA,2013J,31604,F,South East Region,35-55,0,60,Pass
4,AAA,2013J,32885,F,West Midlands Region,0-35,0,60,Pass


In [26]:
# Merge studentVle and vle on 'id_site', 'code_module' and 'code_presentation'
merged_df_vle = pd.merge(student_vle, vle, on=['id_site', 'code_module', 'code_presentation'])
merged_df_vle.head()


,code_module,code_presentation,id_student,id_site,date,sum_click,activity_type,week_from,week_to
0,AAA,2013J,28400,546652,-10,4,forumng,NaN,NaN
1,AAA,2013J,28400,546652,-10,1,forumng,NaN,NaN
2,AAA,2013J,28400,546652,-10,1,forumng,NaN,NaN
3,AAA,2013J,28400,546614,-10,11,homepage,NaN,NaN
4,AAA,2013J,28400,546714,-10,1,oucontent,NaN,NaN


In [27]:
# merge mergedf_df_vle and student_info, so that we have the information combined in one df (number of clicks for each learning material based on code_module and code_presentation)
merged_df = pd.merge(merged_df_vle, student_info, on=['id_student', 'code_module', 'code_presentation'])

In [28]:
merged_df.head()

,code_module,code_presentation,id_student,id_site,date,sum_click,activity_type,week_from,week_to,gender,region,age_band,num_of_prev_attempts,studied_credits,final_result
0,AAA,2013J,28400,546652,-10,4,forumng,NaN,NaN,F,Scotland,35-55,0,60,Pass
1,AAA,2013J,28400,546652,-10,1,forumng,NaN,NaN,F,Scotland,35-55,0,60,Pass
2,AAA,2013J,28400,546652,-10,1,forumng,NaN,NaN,F,Scotland,35-55,0,60,Pass
3,AAA,2013J,28400,546614,-10,11,homepage,NaN,NaN,F,Scotland,35-55,0,60,Pass
4,AAA,2013J,28400,546714,-10,1,oucontent,NaN,NaN,F,Scotland,35-55,0,60,Pass


In [29]:
# drop more columns we do not need
merged_df = merged_df.drop(columns=['id_site', 'date', 'week_from', 'week_to', 'studied_credits', 'num_of_prev_attempts'])
merged_df.head()

,code_module,code_presentation,id_student,sum_click,activity_type,gender,region,age_band,final_result
0,AAA,2013J,28400,4,forumng,F,Scotland,35-55,Pass
1,AAA,2013J,28400,1,forumng,F,Scotland,35-55,Pass
2,AAA,2013J,28400,1,forumng,F,Scotland,35-55,Pass
3,AAA,2013J,28400,11,homepage,F,Scotland,35-55,Pass
4,AAA,2013J,28400,1,oucontent,F,Scotland,35-55,Pass


In [30]:
# Group the merged DataFrame by multiple columns and aggregate the sum of clicks
aggregated_df = merged_df.groupby([
    'code_module',         # Course code
    'code_presentation',   # Presentation code
    'id_student',          # Student ID
    'gender',              # Gender of the student
    'age_band',            # Age band of the student
    'final_result',        # Final result of the student in the course
    'activity_type'        # Type of activity
])['sum_click'].sum().reset_index()  # Sum the clicks and reset the index

aggregated_df.head()



,code_module,code_presentation,id_student,gender,age_band,final_result,activity_type,sum_click
0,AAA,2013J,11391,M,55<=,Pass,forumng,193
1,AAA,2013J,11391,M,55<=,Pass,homepage,138
2,AAA,2013J,11391,M,55<=,Pass,oucontent,553
3,AAA,2013J,11391,M,55<=,Pass,resource,13
4,AAA,2013J,11391,M,55<=,Pass,subpage,32


In [32]:
student_assessment

,id_assessment,id_student,date_submitted,is_banked,score
0,1752,11391,18,0,78.0
1,1752,28400,22,0,70.0
2,1752,31604,17,0,72.0
3,1752,32885,26,0,69.0
4,1752,38053,19,0,79.0
...,...,...,...,...,...
173907,37443,527538,227,0,60.0
173908,37443,534672,229,0,100.0
173909,37443,546286,215,0,80.0
173910,37443,546724,230,0,100.0


In [33]:
# merge assessments and student_assessment
merged_df_assessment = pd.merge(assessments, student_assessment, on=['id_assessment'])
merged_df_assessment

,code_module,code_presentation,id_assessment,assessment_type,date,weight,id_student,date_submitted,is_banked,score
0,AAA,2013J,1752,TMA,19.0,10.0,11391,18,0,78.0
1,AAA,2013J,1752,TMA,19.0,10.0,28400,22,0,70.0
2,AAA,2013J,1752,TMA,19.0,10.0,31604,17,0,72.0
3,AAA,2013J,1752,TMA,19.0,10.0,32885,26,0,69.0
4,AAA,2013J,1752,TMA,19.0,10.0,38053,19,0,79.0
...,...,...,...,...,...,...,...,...,...,...
173907,GGG,2014J,37437,TMA,173.0,0.0,652462,172,0,60.0
173908,GGG,2014J,37437,TMA,173.0,0.0,652539,176,0,75.0
173909,GGG,2014J,37437,TMA,173.0,0.0,653157,187,0,70.0
173910,GGG,2014J,37437,TMA,173.0,0.0,653252,171,0,70.0


In [35]:
# remove not needed columns 
merged_df_assessment = merged_df_assessment.drop(columns=['id_assessment','is_banked'])
merged_df_assessment

,code_module,code_presentation,assessment_type,date,weight,id_student,date_submitted,score
0,AAA,2013J,TMA,19.0,10.0,11391,18,78.0
1,AAA,2013J,TMA,19.0,10.0,28400,22,70.0
2,AAA,2013J,TMA,19.0,10.0,31604,17,72.0
3,AAA,2013J,TMA,19.0,10.0,32885,26,69.0
4,AAA,2013J,TMA,19.0,10.0,38053,19,79.0
...,...,...,...,...,...,...,...,...
173907,GGG,2014J,TMA,173.0,0.0,652462,172,60.0
173908,GGG,2014J,TMA,173.0,0.0,652539,176,75.0
173909,GGG,2014J,TMA,173.0,0.0,653157,187,70.0
173910,GGG,2014J,TMA,173.0,0.0,653252,171,70.0


In [36]:
merged_df_assessment = merged_df_assessment.drop(columns=['date_submitted'])
merged_df_assessment

,code_module,code_presentation,assessment_type,date,weight,id_student,score
0,AAA,2013J,TMA,19.0,10.0,11391,78.0
1,AAA,2013J,TMA,19.0,10.0,28400,70.0
2,AAA,2013J,TMA,19.0,10.0,31604,72.0
3,AAA,2013J,TMA,19.0,10.0,32885,69.0
4,AAA,2013J,TMA,19.0,10.0,38053,79.0
...,...,...,...,...,...,...,...
173907,GGG,2014J,TMA,173.0,0.0,652462,60.0
173908,GGG,2014J,TMA,173.0,0.0,652539,75.0
173909,GGG,2014J,TMA,173.0,0.0,653157,70.0
173910,GGG,2014J,TMA,173.0,0.0,653252,70.0


In [37]:
final_df = pd.merge(aggregated_df, merged_df_assessment, on=['code_module', 'code_presentation', 'id_student'])
final_df

,code_module,code_presentation,id_student,gender,age_band,final_result,activity_type,sum_click,assessment_type,date,weight,score
0,AAA,2013J,11391,M,55<=,Pass,forumng,193,TMA,19.0,10.0,78.0
1,AAA,2013J,11391,M,55<=,Pass,forumng,193,TMA,54.0,20.0,85.0
2,AAA,2013J,11391,M,55<=,Pass,forumng,193,TMA,117.0,20.0,80.0
3,AAA,2013J,11391,M,55<=,Pass,forumng,193,TMA,166.0,20.0,85.0
4,AAA,2013J,11391,M,55<=,Pass,forumng,193,TMA,215.0,30.0,82.0
...,...,...,...,...,...,...,...,...,...,...,...,...
1426890,GGG,2014J,2684003,F,35-55,Distinction,subpage,27,CMA,229.0,0.0,100.0
1426891,GGG,2014J,2684003,F,35-55,Distinction,subpage,27,CMA,229.0,0.0,100.0
1426892,GGG,2014J,2684003,F,35-55,Distinction,subpage,27,TMA,61.0,0.0,80.0
1426893,GGG,2014J,2684003,F,35-55,Distinction,subpage,27,TMA,124.0,0.0,80.0


In [38]:
nan_rows = final_df.loc[final_df['score'].isna()]
nan_rows

,code_module,code_presentation,id_student,gender,age_band,final_result,activity_type,sum_click,assessment_type,date,weight,score
3354,AAA,2013J,260355,F,35-55,Withdrawn,forumng,50,TMA,117.0,20.0,NaN
3357,AAA,2013J,260355,F,35-55,Withdrawn,homepage,139,TMA,117.0,20.0,NaN
3360,AAA,2013J,260355,F,35-55,Withdrawn,oucontent,217,TMA,117.0,20.0,NaN
3363,AAA,2013J,260355,F,35-55,Withdrawn,resource,3,TMA,117.0,20.0,NaN
3366,AAA,2013J,260355,F,35-55,Withdrawn,subpage,14,TMA,117.0,20.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
1411723,GGG,2014J,648110,F,0-35,Withdrawn,forumng,6,TMA,61.0,0.0,NaN
1411724,GGG,2014J,648110,F,0-35,Withdrawn,homepage,31,TMA,61.0,0.0,NaN
1411725,GGG,2014J,648110,F,0-35,Withdrawn,oucontent,12,TMA,61.0,0.0,NaN
1411726,GGG,2014J,648110,F,0-35,Withdrawn,resource,25,TMA,61.0,0.0,NaN


In [39]:
# remove rows where score is NaN (Withdrawn)
final_df = final_df.dropna(subset=['score'])
final_df

,code_module,code_presentation,id_student,gender,age_band,final_result,activity_type,sum_click,assessment_type,date,weight,score
0,AAA,2013J,11391,M,55<=,Pass,forumng,193,TMA,19.0,10.0,78.0
1,AAA,2013J,11391,M,55<=,Pass,forumng,193,TMA,54.0,20.0,85.0
2,AAA,2013J,11391,M,55<=,Pass,forumng,193,TMA,117.0,20.0,80.0
3,AAA,2013J,11391,M,55<=,Pass,forumng,193,TMA,166.0,20.0,85.0
4,AAA,2013J,11391,M,55<=,Pass,forumng,193,TMA,215.0,30.0,82.0
...,...,...,...,...,...,...,...,...,...,...,...,...
1426890,GGG,2014J,2684003,F,35-55,Distinction,subpage,27,CMA,229.0,0.0,100.0
1426891,GGG,2014J,2684003,F,35-55,Distinction,subpage,27,CMA,229.0,0.0,100.0
1426892,GGG,2014J,2684003,F,35-55,Distinction,subpage,27,TMA,61.0,0.0,80.0
1426893,GGG,2014J,2684003,F,35-55,Distinction,subpage,27,TMA,124.0,0.0,80.0


In [40]:
nan_rows = final_df.loc[final_df['date'].isna()]
nan_rows

,code_module,code_presentation,id_student,gender,age_band,final_result,activity_type,sum_click,assessment_type,date,weight,score
337887,CCC,2014B,29764,M,0-35,Distinction,forumng,21,Exam,NaN,100.0,94.0
337896,CCC,2014B,29764,M,0-35,Distinction,homepage,224,Exam,NaN,100.0,94.0
337905,CCC,2014B,29764,M,0-35,Distinction,oucontent,92,Exam,NaN,100.0,94.0
337914,CCC,2014B,29764,M,0-35,Distinction,page,11,Exam,NaN,100.0,94.0
337923,CCC,2014B,29764,M,0-35,Distinction,quiz,2135,Exam,NaN,100.0,94.0
...,...,...,...,...,...,...,...,...,...,...,...,...
708803,DDD,2014J,2689863,F,0-35,Pass,oucollaborate,22,Exam,NaN,100.0,67.0
708810,DDD,2014J,2689863,F,0-35,Pass,oucontent,92,Exam,NaN,100.0,67.0
708817,DDD,2014J,2689863,F,0-35,Pass,resource,43,Exam,NaN,100.0,67.0
708824,DDD,2014J,2689863,F,0-35,Pass,subpage,110,Exam,NaN,100.0,67.0


In [41]:
# final exam rows do not have a date, so we fill them with 0
final_df['date'] = final_df['date'].fillna(0)


/var/folders/f6/5ylgypr54rb9t4hjk829bhmm0000gn/T/ipykernel_2190/1930596183.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  final_df['date'] = final_df['date'].fillna(0)


In [42]:
final_df

,code_module,code_presentation,id_student,gender,age_band,final_result,activity_type,sum_click,assessment_type,date,weight,score
0,AAA,2013J,11391,M,55<=,Pass,forumng,193,TMA,19.0,10.0,78.0
1,AAA,2013J,11391,M,55<=,Pass,forumng,193,TMA,54.0,20.0,85.0
2,AAA,2013J,11391,M,55<=,Pass,forumng,193,TMA,117.0,20.0,80.0
3,AAA,2013J,11391,M,55<=,Pass,forumng,193,TMA,166.0,20.0,85.0
4,AAA,2013J,11391,M,55<=,Pass,forumng,193,TMA,215.0,30.0,82.0
...,...,...,...,...,...,...,...,...,...,...,...,...
1426890,GGG,2014J,2684003,F,35-55,Distinction,subpage,27,CMA,229.0,0.0,100.0
1426891,GGG,2014J,2684003,F,35-55,Distinction,subpage,27,CMA,229.0,0.0,100.0
1426892,GGG,2014J,2684003,F,35-55,Distinction,subpage,27,TMA,61.0,0.0,80.0
1426893,GGG,2014J,2684003,F,35-55,Distinction,subpage,27,TMA,124.0,0.0,80.0


In [43]:
# converitng column from int to str
final_df['id_student'] = final_df['id_student'].apply(lambda x: str(int(x)))


/var/folders/f6/5ylgypr54rb9t4hjk829bhmm0000gn/T/ipykernel_2190/478635318.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  final_df['id_student'] = final_df['id_student'].apply(lambda x: str(int(x)))


## Drop some data
The free tier from Neo4j is limited to a certain amount of relations and nodes. So we need to drop some data

In [44]:
# counting how many unique students are in the dataset:
len(final_df['id_student'].unique())

22679

In [46]:
final_df

,code_module,code_presentation,id_student,gender,age_band,final_result,activity_type,sum_click,assessment_type,date,weight,score
0,AAA,2013J,11391,M,55<=,Pass,forumng,193,TMA,19.0,10.0,78.0
1,AAA,2013J,11391,M,55<=,Pass,forumng,193,TMA,54.0,20.0,85.0
2,AAA,2013J,11391,M,55<=,Pass,forumng,193,TMA,117.0,20.0,80.0
3,AAA,2013J,11391,M,55<=,Pass,forumng,193,TMA,166.0,20.0,85.0
4,AAA,2013J,11391,M,55<=,Pass,forumng,193,TMA,215.0,30.0,82.0
...,...,...,...,...,...,...,...,...,...,...,...,...
1426890,GGG,2014J,2684003,F,35-55,Distinction,subpage,27,CMA,229.0,0.0,100.0
1426891,GGG,2014J,2684003,F,35-55,Distinction,subpage,27,CMA,229.0,0.0,100.0
1426892,GGG,2014J,2684003,F,35-55,Distinction,subpage,27,TMA,61.0,0.0,80.0
1426893,GGG,2014J,2684003,F,35-55,Distinction,subpage,27,TMA,124.0,0.0,80.0


In [52]:
# choose randomly 5000 id_student to drop
unique_students = final_df['id_student'].unique()
random_students = np.random.choice(unique_students, size=5000, replace=False)

random_students_list = random_students.tolist()


In [53]:
filtered_df = final_df[~final_df['id_student'].isin(random_students_list)]
filtered_df

,code_module,code_presentation,id_student,gender,age_band,final_result,activity_type,sum_click,assessment_type,date,weight,score
30,AAA,2013J,28400,F,35-55,Pass,dataplus,10,TMA,19.0,10.0,70.0
31,AAA,2013J,28400,F,35-55,Pass,dataplus,10,TMA,54.0,20.0,68.0
32,AAA,2013J,28400,F,35-55,Pass,dataplus,10,TMA,117.0,20.0,70.0
33,AAA,2013J,28400,F,35-55,Pass,dataplus,10,TMA,166.0,20.0,64.0
34,AAA,2013J,28400,F,35-55,Pass,dataplus,10,TMA,215.0,30.0,60.0
...,...,...,...,...,...,...,...,...,...,...,...,...
1426890,GGG,2014J,2684003,F,35-55,Distinction,subpage,27,CMA,229.0,0.0,100.0
1426891,GGG,2014J,2684003,F,35-55,Distinction,subpage,27,CMA,229.0,0.0,100.0
1426892,GGG,2014J,2684003,F,35-55,Distinction,subpage,27,TMA,61.0,0.0,80.0
1426893,GGG,2014J,2684003,F,35-55,Distinction,subpage,27,TMA,124.0,0.0,80.0


In [60]:
# counting how many unique students are in the dataset:
len(filtered_df['id_student'].unique())


17679

In [55]:
# save new final_df to csv
filtered_df.to_csv("final_df2.csv", index=False)

## Insert data into Neo4j

In [56]:
# Connect to the Neo4j database
URI = "***"
AUTH = ("neo4j", "***")
driver = GraphDatabase.driver(URI, auth=AUTH)


In [61]:
# write in batches

def insert_data(tx, rows):
    query = (
        "UNWIND $rows AS row "
        "MERGE (s:Student {id_student: row.id_student}) "
        "ON CREATE SET s.gender = row.gender, s.age_band = row.age_band "
        "MERGE (c:Course {code_module: row.code_module}) "
        "MERGE (learning:Learning_Material {type: row.activity_type}) "
        "MERGE (s)-[:ENROLLED_IN  { code_presentation: row.code_presentation }]->(c) "
        "MERGE (s)-[:USED  { sum_click : row.sum_click }]->(learning) "
        "MERGE (c)-[:PROVIDES  { code_presentation  : row.code_presentation }]->(learning) "
        "MERGE (g:Grade {assessment_type: row.assessment_type, weight: row.weight, date: row.date}) "
        "MERGE (s)-[:HAS_GRADE  { score  : row.score, final_result : row.final_result }]->(g) "
        "MERGE (c)-[:HAS_ASSESSMENT]->(g) "
        "MERGE (learning)-[:USED_FOR {score: row.score}]->(g)"
    )

    tx.run(query, rows=rows)

batch_size = 1500  # Adjust batch size as needed
with driver.session() as session:
    for i in range(0, len(filtered_df), batch_size):
        batch = filtered_df.iloc[i:i+batch_size].to_dict('records')
        try:
            session.write_transaction(insert_data, batch)
        except Exception as e:
            print(f"Error inserting data for batch starting at row {i}. Error: {str(e)}")
    print("finished")

/var/folders/f6/5ylgypr54rb9t4hjk829bhmm0000gn/T/ipykernel_2190/2116361889.py:22: DeprecationWarning: Using a driver after it has been closed is deprecated. Future versions of the driver will raise an error.
  with driver.session() as session:
/var/folders/f6/5ylgypr54rb9t4hjk829bhmm0000gn/T/ipykernel_2190/2116361889.py:26: DeprecationWarning: write_transaction has been renamed to execute_write
  session.write_transaction(insert_data, batch)
/var/folders/f6/5ylgypr54rb9t4hjk829bhmm0000gn/T/ipykernel_2190/2116361889.py:26: DeprecationWarning: write_transaction has been renamed to execute_write
  session.write_transaction(insert_data, batch)
/var/folders/f6/5ylgypr54rb9t4hjk829bhmm0000gn/T/ipykernel_2190/2116361889.py:26: DeprecationWarning: write_transaction has been renamed to execute_write
  session.write_transaction(insert_data, batch)
/var/folders/f6/5ylgypr54rb9t4hjk829bhmm0000gn/T/ipykernel_2190/2116361889.py:26: DeprecationWarning: write_transaction has been renamed to execute_wr

finished


## Create new csv files for relational database
Since we were forced to reduce our dataset, we also need to adjust the initial csv files used for the relational database

In [18]:
final_df = pd.read_csv('final_df2.csv')
final_df

,code_module,code_presentation,id_student,gender,age_band,final_result,activity_type,sum_click,assessment_type,date,weight,score
0,AAA,2013J,28400,F,35-55,Pass,dataplus,10,TMA,19.0,10.0,70.0
1,AAA,2013J,28400,F,35-55,Pass,dataplus,10,TMA,54.0,20.0,68.0
2,AAA,2013J,28400,F,35-55,Pass,dataplus,10,TMA,117.0,20.0,70.0
3,AAA,2013J,28400,F,35-55,Pass,dataplus,10,TMA,166.0,20.0,64.0
4,AAA,2013J,28400,F,35-55,Pass,dataplus,10,TMA,215.0,30.0,60.0
...,...,...,...,...,...,...,...,...,...,...,...,...
1108837,GGG,2014J,2684003,F,35-55,Distinction,subpage,27,CMA,229.0,0.0,100.0
1108838,GGG,2014J,2684003,F,35-55,Distinction,subpage,27,CMA,229.0,0.0,100.0
1108839,GGG,2014J,2684003,F,35-55,Distinction,subpage,27,TMA,61.0,0.0,80.0
1108840,GGG,2014J,2684003,F,35-55,Distinction,subpage,27,TMA,124.0,0.0,80.0


### Assessments

In [12]:
assessments = pd.read_csv('archive/assessments.csv')
assessments

,code_module,code_presentation,id_assessment,assessment_type,date,weight
0,AAA,2013J,1752,TMA,19.0,10.0
1,AAA,2013J,1753,TMA,54.0,20.0
2,AAA,2013J,1754,TMA,117.0,20.0
3,AAA,2013J,1755,TMA,166.0,20.0
4,AAA,2013J,1756,TMA,215.0,30.0
...,...,...,...,...,...,...
201,GGG,2014J,37443,CMA,229.0,0.0
202,GGG,2014J,37435,TMA,61.0,0.0
203,GGG,2014J,37436,TMA,124.0,0.0
204,GGG,2014J,37437,TMA,173.0,0.0


In [13]:
# Merge the 'assessments' DataFrame with 'final_df' DataFrame on specified columns using an inner join
merged_df = pd.merge(
    assessments,
    final_df,
    on=['code_module', 'code_presentation', 'assessment_type', 'date', 'weight'],
    how='inner'
)
result_df = merged_df[assessments.columns]
result_df

,code_module,code_presentation,id_assessment,assessment_type,date,weight
0,AAA,2013J,1752,TMA,19.0,10.0
1,AAA,2013J,1752,TMA,19.0,10.0
2,AAA,2013J,1752,TMA,19.0,10.0
3,AAA,2013J,1752,TMA,19.0,10.0
4,AAA,2013J,1752,TMA,19.0,10.0
...,...,...,...,...,...,...
2822145,GGG,2014J,37437,TMA,173.0,0.0
2822146,GGG,2014J,37437,TMA,173.0,0.0
2822147,GGG,2014J,37437,TMA,173.0,0.0
2822148,GGG,2014J,37437,TMA,173.0,0.0


In [14]:
assessments_new = result_df.drop_duplicates()
assessments_new

,code_module,code_presentation,id_assessment,assessment_type,date,weight
0,AAA,2013J,1752,TMA,19.0,10.0
2053,AAA,2013J,1753,TMA,54.0,20.0
4011,AAA,2013J,1754,TMA,117.0,20.0
5896,AAA,2013J,1755,TMA,166.0,20.0
7617,AAA,2013J,1756,TMA,215.0,30.0
...,...,...,...,...,...,...
2785710,GGG,2014J,37442,CMA,229.0,0.0
2799830,GGG,2014J,37443,CMA,229.0,0.0
2813950,GGG,2014J,37435,TMA,61.0,0.0
2816848,GGG,2014J,37436,TMA,124.0,0.0


In [16]:
# save new assessments.csv
assessments_new.to_csv('archive/archive_reduced/assessments.csv', index=False)

### StudentAssessments

In [21]:
studentAssessments = pd.read_csv('archive/studentAssessment2.csv')
studentAssessments

,id_assessment,id_student,score
0,1752,11391,78.0
1,1752,28400,70.0
2,1752,31604,72.0
3,1752,32885,69.0
4,1752,38053,79.0
...,...,...,...
173907,37443,527538,60.0
173908,37443,534672,100.0
173909,37443,546286,80.0
173910,37443,546724,100.0


In [22]:
merged_df = pd.merge(
    studentAssessments,
    final_df,
    on=['id_student', 'score'],
    how='inner'
)
result_df = merged_df[studentAssessments.columns]
result_df

,id_assessment,id_student,score
0,1752,28400,70.0
1,1752,28400,70.0
2,1752,28400,70.0
3,1752,28400,70.0
4,1752,28400,70.0
...,...,...,...
1708170,37443,558486,80.0
1708171,37443,558486,80.0
1708172,37443,558486,80.0
1708173,37443,558486,80.0


In [23]:
studentAssessments_new = result_df.drop_duplicates()
studentAssessments_new

,id_assessment,id_student,score
0,1752,28400,70.0
14,1752,31604,72.0
22,1752,32885,69.0
29,1752,38053,79.0
37,1752,45462,70.0
...,...,...,...
1708080,37443,521631,20.0
1708087,37443,534672,100.0
1708101,37443,546286,80.0
1708115,37443,546724,100.0


In [24]:
studentAssessments_new.to_csv('archive/archive_reduced/studentAssessments.csv', index=False)

### StudentInfo

In [28]:

studentInfo = pd.read_csv('archive/archive2/studentInfo2.csv')
studentInfo

,code_module,code_presentation,id_student,gender,final_result
0,AAA,2013J,11391,M,Pass
1,AAA,2013J,28400,F,Pass
2,AAA,2013J,30268,F,Withdrawn
3,AAA,2013J,31604,F,Pass
4,AAA,2013J,32885,F,Pass
...,...,...,...,...,...
28780,GGG,2014J,2640965,F,Fail
28781,GGG,2014J,2645731,F,Distinction
28782,GGG,2014J,2648187,F,Pass
28783,GGG,2014J,2679821,F,Withdrawn


In [29]:
merged_df = pd.merge(
    studentInfo,
    final_df,
    on=['code_module', 'code_presentation', 'id_student', 'gender', 'final_result'],
    how='inner'
)
result_df = merged_df[studentInfo.columns]
result_df

,code_module,code_presentation,id_student,gender,final_result
0,AAA,2013J,28400,F,Pass
1,AAA,2013J,28400,F,Pass
2,AAA,2013J,28400,F,Pass
3,AAA,2013J,28400,F,Pass
4,AAA,2013J,28400,F,Pass
...,...,...,...,...,...
1108837,GGG,2014J,2684003,F,Distinction
1108838,GGG,2014J,2684003,F,Distinction
1108839,GGG,2014J,2684003,F,Distinction
1108840,GGG,2014J,2684003,F,Distinction


In [30]:
studentInfo_new = result_df.drop_duplicates()
studentInfo_new

,code_module,code_presentation,id_student,gender,final_result
0,AAA,2013J,28400,F,Pass
35,AAA,2013J,31604,F,Pass
75,AAA,2013J,32885,F,Pass
110,AAA,2013J,38053,M,Pass
150,AAA,2013J,45462,M,Pass
...,...,...,...,...,...
1108608,GGG,2014J,2620947,F,Distinction
1108671,GGG,2014J,2645731,F,Distinction
1108734,GGG,2014J,2648187,F,Pass
1108788,GGG,2014J,2679821,F,Withdrawn


In [31]:
studentInfo_new.to_csv("archive/archive_reduced/studentInfo.csv", index=False)

In [7]:
studentVle = pd.read_csv('archive/archive2/studentVle2.csv')
studentVle


,code_module,code_presentation,id_student,id_site,sum_click
0,AAA,2013J,28400,546652,4
1,AAA,2013J,28400,546652,1
2,AAA,2013J,28400,546652,1
3,AAA,2013J,28400,546614,11
4,AAA,2013J,28400,546714,1
...,...,...,...,...,...
10655275,GGG,2014J,675811,896943,3
10655276,GGG,2014J,675578,896943,1
10655277,GGG,2014J,654064,896943,3
10655278,GGG,2014J,654064,896939,1


In [14]:
df_filtered2.to_csv('archive/archive_reduced/studentVle2.csv', index=False)

In [8]:
# Filter studentvle to keep only the rows whose id_student is contained in final_df
filtered_studentvle = studentVle[studentVle['id_student'].isin(final_df['id_student'])]

filtered_studentvle

,code_module,code_presentation,id_student,id_site,sum_click
0,AAA,2013J,28400,546652,4
1,AAA,2013J,28400,546652,1
2,AAA,2013J,28400,546652,1
3,AAA,2013J,28400,546614,11
4,AAA,2013J,28400,546714,1
...,...,...,...,...,...
10655273,GGG,2014J,644226,896943,1
10655275,GGG,2014J,675811,896943,3
10655277,GGG,2014J,654064,896943,3
10655278,GGG,2014J,654064,896939,1


In [23]:
filtered_studentvle.to_csv('archive/archive_reduced/studentVle.csv', index=False)

### VLE

In [38]:
vle = pd.read_csv('archive/vle.csv')
vle

,id_site,code_module,code_presentation,activity_type
0,546943,AAA,2013J,resource
1,546712,AAA,2013J,oucontent
2,546998,AAA,2013J,resource
3,546888,AAA,2013J,url
4,547035,AAA,2013J,resource
...,...,...,...,...
6359,897063,GGG,2014J,resource
6360,897109,GGG,2014J,resource
6361,896965,GGG,2014J,oucontent
6362,897060,GGG,2014J,resource


In [39]:
merged_df = pd.merge(
    vle,
    final_df,
    on=['code_module', 'code_presentation','activity_type'],
    how='inner'
)
result_df = merged_df[vle.columns]
result_df

,id_site,code_module,code_presentation,activity_type
0,546943,AAA,2013J,resource
1,546943,AAA,2013J,resource
2,546943,AAA,2013J,resource
3,546943,AAA,2013J,resource
4,546943,AAA,2013J,resource
...,...,...,...,...
40465116,897100,GGG,2014J,resource
40465117,897100,GGG,2014J,resource
40465118,897100,GGG,2014J,resource
40465119,897100,GGG,2014J,resource


In [41]:
vle_new = result_df.drop_duplicates()
vle_new

,id_site,code_module,code_presentation,activity_type
0,546943,AAA,2013J,resource
1303,546712,AAA,2013J,oucontent
2613,546998,AAA,2013J,resource
3916,546888,AAA,2013J,url
5187,547035,AAA,2013J,resource
...,...,...,...,...
40447938,897063,GGG,2014J,resource
40451374,897109,GGG,2014J,resource
40454810,896965,GGG,2014J,oucontent
40458249,897060,GGG,2014J,resource


In [42]:
vle_new.to_csv('archive/archive_reduced/vle.csv', index=False)

# Delete everything from the Graph Database

In [57]:
with GraphDatabase.driver(URI, auth=AUTH) as driver:
    with driver.session() as session:
       session.run("MATCH (n) DETACH DELETE n")

Failed to read from defunct connection IPv4Address(('71451be8.databases.neo4j.io', 7687)) (ResolvedIPv4Address(('34.78.76.49', 7687)))


In [ ]:
# Close the driver connection
driver.close()